# 04.6 — Analogy Evaluation & Intrinsic Metrics

**Problem it solves:** How do you know if your word vectors are good? Intrinsic evaluation measures vector geometry directly, before any downstream task.

**Word analogy (Mikolov 2013):** 'king - man + woman = queen'. Encoded as: find w where cos(w, v_queen) is maximum, given v_king - v_man + v_woman.

**Important lesson:** Good analogy scores don't guarantee good downstream performance. Always evaluate extrinsically too.

---

In [ ]:
import numpy as np
from collections import defaultdict

def cosine_sim(a, b):
    n_a, n_b = np.linalg.norm(a), np.linalg.norm(b)
    if n_a == 0 or n_b == 0: return 0.0
    return float(np.dot(a, b) / (n_a * n_b))

class EmbeddingEvaluator:
    """
    Intrinsic evaluation for word embeddings:
    1. Word similarity (compare to human ratings)
    2. Word analogy (3CosAdd method)
    3. Nearest neighbors (qualitative)
    """
    
    def __init__(self, vectors: np.ndarray, vocab: list):
        self.vectors = vectors
        self.vocab = vocab
        self.word2idx = {w: i for i, w in enumerate(vocab)}
        
        # Precompute normalized vectors for fast cosine similarity
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        norms[norms == 0] = 1
        self.normed = vectors / norms
    
    def get_vec(self, word: str) -> np.ndarray:
        if word not in self.word2idx:
            return np.zeros(self.vectors.shape[1])
        return self.vectors[self.word2idx[word]]
    
    def analogy_3cosadd(self, a: str, b: str, c: str, n=5) -> list:
        """
        3CosAdd: find d such that d - c ≈ b - a
        i.e., king - man + woman ≈ queen
        
        Method: maximize cos(d, b-a+c) over all vocab words d
        excluding a, b, c themselves.
        """
        va = self.get_vec(a)
        vb = self.get_vec(b)
        vc = self.get_vec(c)
        
        target = vb - va + vc
        target_norm = np.linalg.norm(target)
        if target_norm == 0:
            return []
        target = target / target_norm
        
        sims = self.normed @ target
        exclude = {a, b, c}
        
        top = sims.argsort()[::-1]
        results = []
        for i in top:
            word = self.vocab[i]
            if word not in exclude:
                results.append((word, float(sims[i])))
            if len(results) == n:
                break
        return results
    
    def evaluate_analogies(self, analogies: list[tuple]) -> dict:
        """
        Evaluate on a list of analogies: [(a, b, c, d), ...]
        where b - a + c should ≈ d
        """
        correct = 0
        top5_correct = 0
        skipped = 0
        
        for a, b, c, d in analogies:
            # Skip if any word not in vocab
            if not all(w in self.word2idx for w in [a, b, c, d]):
                skipped += 1
                continue
            
            results = self.analogy_3cosadd(a, b, c, n=5)
            top5_words = [w for w, _ in results]
            
            if results and results[0][0] == d:
                correct += 1
            if d in top5_words:
                top5_correct += 1
        
        total_valid = len(analogies) - skipped
        return {
            'top1_accuracy': correct / total_valid if total_valid > 0 else 0,
            'top5_accuracy': top5_correct / total_valid if total_valid > 0 else 0,
            'total': len(analogies),
            'valid': total_valid,
            'skipped': skipped,
        }
    
    def word_similarity(self, pairs: list[tuple]) -> float:
        """
        Word similarity: correlate cosine similarity with human ratings.
        pairs: [(word1, word2, human_score), ...]
        Returns Spearman rank correlation.
        """
        model_scores = []
        human_scores = []
        
        for w1, w2, human in pairs:
            if w1 not in self.word2idx or w2 not in self.word2idx:
                continue
            v1, v2 = self.get_vec(w1), self.get_vec(w2)
            model_scores.append(cosine_sim(v1, v2))
            human_scores.append(human)
        
        if len(model_scores) < 2:
            return 0.0
        
        # Spearman rank correlation
        def rank(lst):
            sorted_idx = np.argsort(lst)
            ranks = np.empty_like(sorted_idx)
            ranks[sorted_idx] = np.arange(len(lst))
            return ranks.astype(float)
        
        r_model = rank(model_scores)
        r_human = rank(human_scores)
        n = len(r_model)
        r_model -= r_model.mean()
        r_human -= r_human.mean()
        denom = np.linalg.norm(r_model) * np.linalg.norm(r_human)
        return float(np.dot(r_model, r_human) / denom) if denom > 0 else 0.0
    
    def nearest_neighbors(self, word: str, n=8) -> list:
        if word not in self.word2idx:
            return []
        vec = self.normed[self.word2idx[word]]
        sims = self.normed @ vec
        top = sims.argsort()[::-1][1:n+1]
        return [(self.vocab[i], float(sims[i])) for i in top]

# Create dummy vectors to demo the evaluation framework
np.random.seed(42)
vocab = ['king', 'queen', 'man', 'woman', 'prince', 'princess', 
         'france', 'paris', 'germany', 'berlin', 'england', 'london',
         'cat', 'dog', 'animal', 'pet',
         'big', 'large', 'small', 'tiny']

# Simulate 'good' embeddings by adding relational structure
dim = 50
vectors = np.random.randn(len(vocab), dim) * 0.1

# Add gender dimension
gender_dir = np.random.randn(dim) * 0.5
for w in ['king', 'man', 'prince', 'brother']:
    if w in vocab:
        vectors[vocab.index(w)] += gender_dir
for w in ['queen', 'woman', 'princess', 'sister']:
    if w in vocab:
        vectors[vocab.index(w)] -= gender_dir

# Add royalty dimension
royal_dir = np.random.randn(dim) * 0.5
for w in ['king', 'queen', 'prince', 'princess']:
    if w in vocab:
        vectors[vocab.index(w)] += royal_dir

evaluator = EmbeddingEvaluator(vectors, vocab)

analogies = [
    ('man', 'king', 'woman', 'queen'),       # classic gender-royalty
    ('man', 'prince', 'woman', 'princess'),
    ('france', 'paris', 'germany', 'berlin'),
    ('france', 'paris', 'england', 'london'),
    ('big', 'large', 'small', 'tiny'),
]

print("Word analogies (a - b + c = d?):")
for a, b, c, expected in analogies:
    results = evaluator.analogy_3cosadd(a, b, c, n=3)
    predicted = results[0][0] if results else '?'
    correct = '✓' if predicted == expected else '✗'
    print(f"  {a}-{b}+{c}={expected}  predicted={predicted} {correct}")
    print(f"    top 3: {results}")

In [ ]:
# Intrinsic vs Extrinsic: the disconnect

print("=== The Intrinsic vs Extrinsic Disconnect ===")
print()
print("Intrinsic eval: measures geometry of the embedding space")
print("  - Word similarity (SimLex-999, WordSim-353)")
print("  - Word analogies (Google Analogy Test, BATS)")
print()
print("Extrinsic eval: measures performance on downstream tasks")
print("  - NER, POS, sentiment, NLI, etc.")
print()
print("Key findings (Faruqui et al., 2016):")
print("  - Analogy accuracy ≈ 0 doesn't mean the embeddings are bad")
print("  - High analogy accuracy sometimes HURTS downstream tasks")
print("  - Domain-specific training corpus beats generic large corpus")
print("    for domain-specific downstream tasks")
print()
print("Rule: always run extrinsic evaluation. Analogies are a quick sanity check,")
print("not a reliable predictor of downstream performance.")

# SimLex-999 style similarity: examples
simlex_sample = [
    ('cat', 'dog', 7.08),      # similar concepts
    ('cat', 'animal', 7.22),   # hypernym — less similar in SimLex (assoc ≠ sim)
    ('big', 'large', 9.47),    # near-synonyms
    ('big', 'small', 0.88),    # antonyms — low similarity in SimLex!
]
print("\nSimLex-999 examples (SimLex measures SIMILARITY, not RELATEDNESS):")
for w1, w2, score in simlex_sample:
    print(f"  {w1:10} - {w2:10}: {score:.2f}/10")
print("Note: antonyms are RELATED but not SIMILAR — SimLex distinguishes this")

## Summary

**Word analogy evaluation** (king - man + woman = queen) is a famous demo but a weak metric. It:
- Only works if all 4 words are in vocab
- Favors frequent words
- Doesn't predict downstream task performance

**Better intrinsic metrics:**
- QVEC: correlation with linguistic features
- SimLex-999: true similarity (not association)
- Downstream task performance (extrinsic)

**The real test:** How does the embedding improve your task? Use embedding as features in your model and compare to the baseline.

---
**Module 04 complete. Next: Module 05 — Sequence Models**